In [35]:
import pandas as pd
import pickle
from pathlib import Path
import numpy as np


from sklearn.model_selection import KFold

from surprise.accuracy import rmse
import warnings
warnings.filterwarnings('ignore')

# Đường dẫn
BASE_DIR = Path.cwd().parent.parent

DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "ml" / "model"

# Hyperparameters
K = 16          # số latent factors (embedding dim)
lambda_reg = 0.01  # regularization
lr = 0.005      # learning rate
epochs = 30     # số epoch
n_folds = 5     # 5-fold CV

In [36]:
#LOAD DATA
print("Đang load real ratings...")
ratings = pd.read_csv(DATA_PROCESSED / 'user_item_ratings_real.csv')
# ratings = pd.read_csv(
#     r"recommend\backend\data\processed\user_item_ratings_real.csv"
# )

# Map user_id và product_id thành index số
user_ids = ratings['user_id'].unique()
item_ids = ratings['product_id'].unique()

user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
item_to_idx = {iid: i for i, iid in enumerate(item_ids)}

ratings['user_idx'] = ratings['user_id'].map(user_to_idx)
ratings['item_idx'] = ratings['product_id'].map(item_to_idx)

n_users = len(user_ids)
n_items = len(item_ids)

print(f"Users: {n_users} | Items: {n_items} | Ratings: {len(ratings)}")

Đang load real ratings...
Users: 11684 | Items: 564 | Ratings: 26515


In [37]:
#HÀM PER-USER 5-FOLD CV
def per_user_kfold_indices(df, n_folds=5, random_state=42):
    # """
    # Step 1: Per-user split.
    # Đảm bảo các rating của cùng 1 user được phân bổ đều vào các fold.
    # Điều này tránh rò rỉ thông tin khi validation (không có user lạ trong validation set).
    # """
    np.random.seed(random_state)
    fold_indices = [[] for _ in range(n_folds)]
    
    for user_idx, group in df.groupby('user_idx'):
        indices = group.index.values
        np.random.shuffle(indices)
        splits = np.array_split(indices, n_folds)
        for fold in range(n_folds):
            fold_indices[fold].extend(splits[fold])
            
    folds = []
    all_indices = np.array(df.index.values)
    for fold in range(n_folds):
        val_idx = np.array(fold_indices[fold])
        train_idx = np.setdiff1d(all_indices, val_idx)
        folds.append((train_idx, val_idx))
    return folds


In [38]:
#svd training function
def train_svd(train_data, val_data, K, epochs, lr, lambda_reg):
    # """
    # Huấn luyện Matrix Factorization bằng Stochastic Gradient Descent (SGD).
    # """
    # Khởi tạo ma trận latent factors W (user) và H (item)
    W = np.random.normal(0, 0.1, (n_users, K))  # User embeddings
    H = np.random.normal(0, 0.1, (n_items, K))  # Item embeddings
    
    train_users = train_data['user_idx'].values.astype(int)
    train_items = train_data['item_idx'].values.astype(int)
    train_ratings = train_data['rating'].values
    
    val_users = val_data['user_idx'].values.astype(int)
    val_items = val_data['item_idx'].values.astype(int)
    val_ratings = val_data['rating'].values
    
    for epoch in range(epochs):
        # Shuffle dữ liệu mỗi epoch để SGD hiệu quả hơn
        indices = np.arange(len(train_users))
        np.random.shuffle(indices)
        
        for idx in indices:
            u = train_users[idx]
            i = train_items[idx]
            r = train_ratings[idx]
            
            # Dự đoán (dot product - Ý tưởng cốt lõi)
            pred = np.dot(W[u], H[i])
            
            # Tính sai số e = r - r̂
            err = r - pred
            
            # Cập nhật W và H theo công thức Gradient Descent (có regularization)
            # W_new = W_old + lr * (e * H - λ * W)
            # H_new = H_old + lr * (e * W - λ * H)
            w_old = W[u].copy()
            W[u] += lr * (err * H[i] - lambda_reg * W[u])
            H[i] += lr * (err * w_old - lambda_reg * H[i])
        
        # Đánh giá RMSE trên train và validation mỗi 5 epochs
        if epoch % 5 == 0 or epoch == epochs - 1:
            train_preds = np.sum(W[train_users] * H[train_items], axis=1)
            train_rmse = np.sqrt(np.mean((train_ratings - train_preds) ** 2))
            
            val_preds = np.sum(W[val_users] * H[val_items], axis=1)
            val_rmse = np.sqrt(np.mean((val_ratings - val_preds) ** 2))
            
            print(f"Epoch {epoch+1:2d}/{epochs} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")
    
    final_val_preds = np.sum(W[val_users] * H[val_items], axis=1)
    final_val_rmse = np.sqrt(np.mean((val_ratings - final_val_preds) ** 2))
    
    return W, H, final_val_rmse

In [39]:
#5-FOLD CROSS VALIDATION
print(f"\nBắt đầu {n_folds}-fold Cross Validation...")
# Gọi hàm per_user_kfold_indices để tạo folds
folds = per_user_kfold_indices(ratings, n_folds=n_folds, random_state=42)

best_val_rmse = float('inf')
best_P = None
best_Q = None
best_fold = -1

for fold, (train_idx, val_idx) in enumerate(folds): 
    print(f"\n--- Fold {fold+1}/{n_folds} ---")
    
    train_data = ratings.iloc[train_idx]
    val_data = ratings.iloc[val_idx]
    
    # Train model cho fold hiện tại
    P, Q, final_val_rmse = train_svd(train_data, val_data, K, epochs, lr, lambda_reg)
    print(f"-> Kết quả Fold {fold+1} Validation RMSE: {final_val_rmse:.4f}")
    
    if final_val_rmse < best_val_rmse:
        best_val_rmse = final_val_rmse
        best_P = P.copy()
        best_Q = Q.copy()
        best_fold = fold + 1

print(f"\nHoàn thành!")
print(f"Fold tốt nhất: Fold {best_fold} với Validation RMSE: {best_val_rmse:.4f}")



Bắt đầu 5-fold Cross Validation...

--- Fold 1/5 ---
Epoch  1/30 | Train RMSE: 3.0978 | Val RMSE: 4.4605
Epoch  6/30 | Train RMSE: 2.5776 | Val RMSE: 4.3706
Epoch 11/30 | Train RMSE: 1.9547 | Val RMSE: 4.2612
Epoch 16/30 | Train RMSE: 1.5005 | Val RMSE: 4.1794
Epoch 21/30 | Train RMSE: 1.1989 | Val RMSE: 4.1256
Epoch 26/30 | Train RMSE: 0.9908 | Val RMSE: 4.0915
Epoch 30/30 | Train RMSE: 0.8773 | Val RMSE: 4.0738
-> Kết quả Fold 1 Validation RMSE: 4.0738

--- Fold 2/5 ---
Epoch  1/30 | Train RMSE: 3.9156 | Val RMSE: 3.6234
Epoch  6/30 | Train RMSE: 3.5512 | Val RMSE: 3.2162
Epoch 11/30 | Train RMSE: 2.7476 | Val RMSE: 2.5625
Epoch 16/30 | Train RMSE: 1.8516 | Val RMSE: 2.0036
Epoch 21/30 | Train RMSE: 1.2400 | Val RMSE: 1.6702
Epoch 26/30 | Train RMSE: 0.8845 | Val RMSE: 1.5095
Epoch 30/30 | Train RMSE: 0.7127 | Val RMSE: 1.4336
-> Kết quả Fold 2 Validation RMSE: 1.4336

--- Fold 3/5 ---
Epoch  1/30 | Train RMSE: 3.9623 | Val RMSE: 3.0870
Epoch  6/30 | Train RMSE: 3.5877 | Val RMSE: 2

In [40]:
#save model
model_data = {
    'P': best_P,
    'Q': best_Q,
    'user_to_idx': user_to_idx,
    'item_to_idx': item_to_idx,
    'user_ids': user_ids,
    'item_ids': item_ids,
    'K': K
}

with open(MODEL_DIR / 'svd_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(f"Model đã được lưu tại: {MODEL_DIR / 'svd_model.pkl'}")

Model đã được lưu tại: e:\multiproject\multiplatform\travil\recommend\backend\ml\model\svd_model.pkl


In [41]:
#TEST DỰ ĐOÁN
print(f"\n{'='*50}")
print("TEST DỰ ĐOÁN CHO USER ĐẦU TIÊN")
print(f"{'='*50}")

# Lấy user đầu tiên để test
test_user_idx = 0
test_user_id = user_ids[test_user_idx]

# Lấy các item user đã rating
user_ratings = ratings[ratings['user_idx'] == test_user_idx]
print(f"\nUser {test_user_id} đã rating {len(user_ratings)} sản phẩm:")

# Dự đoán cho tất cả item
predictions = []
for item_idx in range(n_items):
    pred = np.dot(best_P[test_user_idx], best_Q[item_idx])
    predictions.append(pred)

# Lấy top 5 item có dự đoán cao nhất
top_items = np.argsort(predictions)[-5:][::-1]
print(f"\nTop 5 sản phẩm gợi ý cho user {test_user_id}:")
for i, item_idx in enumerate(top_items, 1):
    item_id = item_ids[item_idx]
    pred_score = predictions[item_idx]
    
    # Kiểm tra xem user đã rating item này chưa
    rated = item_id in user_ratings['product_id'].values
    status = " ĐÃ RATING" if rated else " GỢI Ý MỚI"
    
    print(f"  {i}. Item {item_id} | Score: {pred_score:.4f} | {status}")


TEST DỰ ĐOÁN CHO USER ĐẦU TIÊN

User Klook User đã rating 9125 sản phẩm:

Top 5 sản phẩm gợi ý cho user Klook User:
  1. Item 1055 | Score: 5.5967 |  ĐÃ RATING
  2. Item 1311 | Score: 5.4142 |  ĐÃ RATING
  3. Item 1276 | Score: 5.3926 |  ĐÃ RATING
  4. Item 1292 | Score: 5.3857 |  ĐÃ RATING
  5. Item 699 | Score: 5.3819 |  ĐÃ RATING


In [42]:
# #SVD TRAINING
# def train_svd(train_data, val_data, K=16, epochs=30, lr=0.005, lambda_reg=0.01):
#     # Khởi tạo ma trận latent factors
#     P = np.random.normal(0, 0.1, (n_users, K))  # User embeddings
#     Q = np.random.normal(0, 0.1, (n_items, K))  # Item embeddings
    
#     train_rmse_list = []
#     val_rmse_list = []
    
#     for epoch in range(epochs):
#         # Train
#         for _, row in train_data.iterrows():
#             u = int(row['user_idx'])
#             i = int(row['item_idx'])
#             r = row['rating']
            
#             # Dự đoán
#             pred = np.dot(P[u], Q[i])
            
#             # Error
#             err = r - pred
            
#             # Cập nhật
#             P[u] += lr * (err * Q[i] - lambda_reg * P[u])
#             Q[i] += lr * (err * P[u] - lambda_reg * Q[i])
        
#         # Đánh giá (validation)
#         train_pred = np.array([np.dot(P[int(r['user_idx'])], Q[int(r['item_idx'])]) 
#                               for _, r in train_data.iterrows()])
#         val_pred = np.array([np.dot(P[int(r['user_idx'])], Q[int(r['item_idx'])]) 
#                             for _, r in val_data.iterrows()])
        
#         train_rmse = np.sqrt(np.mean((train_data['rating'].values - train_pred)**2))
#         val_rmse = np.sqrt(np.mean((val_data['rating'].values - val_pred)**2))
        
#         train_rmse_list.append(train_rmse)
#         val_rmse_list.append(val_rmse)
        
#         if epoch % 5 == 0 or epoch == epochs-1:
#             print(f"Epoch {epoch+1:2d} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")
    
#     return P, Q, val_rmse_list[-1]